# Tool · 모델의 요청과 Python 실행 · v1.03

**2일차 3교시 · 수업 50분 + 휴식/전환 10분**

**목표:** 현재 장비 정보를 도구로 읽고 도구 호출 한 주기를 직접 추적합니다.

[전체 안내](README.md) · [Python 코드 읽기](../04.LLM_v2/PYTHON_GUIDE.md)

슬라이드로 개념을 먼저 보고 이 노트북을 실행합니다. **필수** 절을 위에서 아래로 실행하세요. 커널은 **Python (Day2 LangChain)**입니다. 다른 노트북의 변수는 자동으로 전달되지 않습니다.

셀 왼쪽 `[*]`는 실행 중이라는 뜻입니다. 응답을 기다린 뒤 다음 셀을 실행하세요. 요청 셀을 재실행하면 서버에 다시 요청합니다. 오류가 나면 그 셀에서 멈추고 오류 원문과 입력을 확인합니다.

**대응 슬라이드:** 25~29쪽 · [PDF 열기](../슬라이드/강의자료_2일차_v2.pdf#page=25)


## 준비 · 경로와 모델 객체

`Path`는 파일 경로를 다루는 도구입니다. 아래 셀은 현재 폴더부터 상위 폴더를 살펴 교재 위치를 찾습니다. `sys.path`에는 교재의 Python 모듈 위치를 추가합니다. 준비 코드는 제공하며 직접 외우지 않아도 됩니다.


In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

# 현재 폴더에 교재가 없으면 부모 폴더를 차례로 확인합니다.
ROOT = Path.cwd().resolve()
while not (ROOT / "04.LLM_v2").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("NEPES 교재 폴더 안에서 노트북을 여세요.")
    ROOT = ROOT.parent
# import가 교재의 공통 모듈을 찾을 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(ROOT / "04.LLM_v2"))
sys.path.insert(0, str(ROOT / "05.VLM_v2"))
print("교재 위치:", ROOT)
print("현재 Python:", sys.executable)


### 사용 전 API 안내 · 모델과 메시지

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `ChatOllama` | `model = ChatOllama(model=이름, base_url=주소, temperature=0)` | Ollama 연결 객체. 생성만으로 추론하지 않습니다. `num_ctx`는 문맥 한도, `num_predict`는 출력 한도입니다. |
| `SystemMessage`, `HumanMessage`, `AIMessage` | `HumanMessage(content="질문")` | 역할별 메시지 객체. 시스템 지침 / 사용자 입력 / 모델 응답을 구분합니다. |
| `model.invoke` | `reply = model.invoke([HumanMessage("질문")])` | 메시지 목록 → `AIMessage`. 이때 서버에 요청합니다. `reply.content`는 본문, `response_metadata`는 실행 정보입니다. |

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# model에는 가중치가 아니라 서버 요청을 위한 Python 객체가 들어갑니다.
# 생성자만 실행해도 대화가 시작되는 것은 아닙니다. invoke가 요청합니다.
model = ChatOllama(
    model="gemma4:e2b-it-qat",
    base_url="http://127.0.0.1:11434",  # 수업은 host network를 사용합니다.
    temperature=0,                     # 비교 조건을 고정합니다. 정답 보장은 아닙니다.
    reasoning=False,                   # 수업에서는 최종 답변을 관찰합니다.
    num_ctx=4096,                      # 문맥 창 크기 설정: 글자 수와 다릅니다.
    num_predict=512,                   # 출력 토큰 상한: 잘리면 상한과 입력을 확인합니다.
    client_kwargs={"timeout": 180},    # 서버 응답을 기다리는 최대 시간(초).
)
print("사용할 모델:", model.model)


## 1. 함수와 Tool의 관계

Python 함수는 입력을 받아 코드를 실행하고 결과를 반환합니다. Tool은 모델이 사용할 수 있도록 함수의 이름·설명·입력 형식을 제공한 것입니다.

이번 도구는 온도와 메모리를 조회합니다. 아래 셀에서 조회 함수의 전체 구현을 읽고 직접 실행합니다. 같은 구현은 5교시 재사용을 위해 [jetson_tools.py](llm_v2/jetson_tools.py)에도 제공합니다. 센서 이름을 확인해 읽고, 읽지 못한 값은 errors에 남깁니다. GPU 사용률이나 전력은 이 함수가 측정하지 않습니다.

![1. 함수와 Tool의 관계](assets/lesson-3-11.svg)


## 직접 Tool을 만드는 순서와 확인할 점

| 순서 | 실제 작업 | 통과 기준 |
|---|---|---|
| 1. 목적·범위 정하기 | “현재 온도·메모리 조회”처럼 한 기능으로 정합니다. | 조회와 정상/이상 판단을 구분합니다. |
| 2. 일반 Python 함수 만들기 | 아래 `read_system_status`에서 파일을 읽고 사전을 만듭니다. | 모델 없이 실행해서 값·단위·측정 시각을 확인합니다. |
| 3. 실패 입력 확인하기 | 센서 경로가 없거나 숫자를 읽지 못하는 경우를 확인합니다. | 없는 값을 0으로 만들지 않고 errors에 기록합니다. |
| 4. Tool로 감싸기 | `get_jetson_status`에 타입 힌트·docstring·`@tool`을 붙입니다. | 모델에게 노출할 인자만 args에 있고, `.invoke({})`가 작동합니다. |
| 5. 모델과 연결하기 | `bind_tools` 후 요청 이름·인자·ID를 확인합니다. | Python은 등록된 도구만 실행하고 결과를 같은 ID로 돌려줍니다. |
| 6. 최종 답변 확인하기 | 측정값·오류와 답변을 나란히 비교합니다. | 측정하지 않은 GPU 사용률이나 정상 온도를 생성하지 않습니다. |

함수를 정의하는 `def` 단계에는 파일을 읽지 않습니다. `read_system_status()` 또는 Tool의 `.invoke({})`를 실행할 때 읽습니다. `get_jetson_status` 안에 모든 코드를 넣을 수도 있지만, **조회 함수와 모델용 Tool을 분리하면 모델 없이 검사하고 다음 교시에서 재사용하기 쉽습니다.** 두 함수 모두 도구를 구현하는 코드의 일부입니다.

현재 수업은 읽기 전용 도구입니다. 추가 도구를 만들 때는 입력 경로 범위, 결과 크기, 외부 요청 제한 시간을 먼저 정하세요. 파일·문서의 내용은 근거 자료이며 실행 지시로 취급하지 않습니다.

### 사용 전 API 안내 · 시스템 파일 조회

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `datetime.now` / `isoformat` | `datetime.now(timezone.utc).isoformat()` | UTC 측정 시각 문자열. |
| `Path.glob` / `read_text` | `Path(thermal_root).glob("thermal_zone*")` | 센서 경로를 순회하고 각 type·temp 파일을 읽습니다. |
| `platform.machine` | `platform.machine()` | 현재 실행 환경의 CPU 아키텍처 문자열. |
| `read_system_status` (직접 작성) | `read_system_status()` | 인자 생략 시 /sys와 /proc 조회 → 사전. 테스트에서는 임시 경로를 전달할 수 있습니다. |

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import platform


def read_system_status(thermal_root="/sys/class/thermal", meminfo_path="/proc/meminfo"):
    """현재 Python 프로세스에 보이는 센서와 메모리를 읽어 사전으로 반환합니다.

    thermal_zone 번호는 장비마다 다르므로 type 파일의 센서 이름을 함께 읽습니다.
    읽지 못한 값은 0으로 채우지 않고 errors에 남깁니다. 이 함수는 정상/이상을
    판정하지 않습니다. 컨테이너에서 보이는 값과 호스트 전체 상태는 구분해야 합니다.
    """
    # 결과를 담을 사전입니다. 빈 사전·목록은 센서별 값과 오류를 모으는 용도입니다.
    result = {
        "measured_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": "live",
        "scope": "현재 Python 실행 환경에 노출된 /sys 및 /proc",
        "machine": platform.machine(),
        "temperatures_c": {},
        "memory_mib": {},
        "errors": [],
    }
    # glob은 조건에 맞는 경로를 찾습니다. sorted는 출력 순서를 일정하게 만듭니다.
    zones = sorted(Path(thermal_root).glob("thermal_zone*"))
    if not zones:
        result["errors"].append("온도 센서 경로가 보이지 않습니다.")
    for zone in zones:
        try:
            name = (zone / "type").read_text().strip()
            # Linux thermal ABI의 정수는 밀리섭씨이므로 1000으로 나눕니다.
            value = int((zone / "temp").read_text().strip()) / 1000
            result["temperatures_c"][f"{zone.name}:{name}"] = value
        except (OSError, ValueError) as exc:
            result["errors"].append(f"{zone.name}: {type(exc).__name__}")
    try:
        values = {}
        for line in Path(meminfo_path).read_text().splitlines():
            # 예: "MemTotal: 16000000 kB"를 항목 이름과 값 부분으로 나눕니다.
            key, text = line.split(":", 1)
            # 공백으로 나눈 첫 항목만 정수로 바꿉니다. 뒤의 단위 kB는 제외합니다.
            values[key] = int(text.split()[0])
        # /proc/meminfo의 kB는 여기서 1024로 나누어 MiB로 표시합니다.
        result["memory_mib"]["total"] = round(values["MemTotal"] / 1024, 1)
        result["memory_mib"]["available"] = round(values["MemAvailable"] / 1024, 1)
    except (OSError, ValueError, KeyError, IndexError) as exc:
        result["errors"].append(f"메모리 조회: {type(exc).__name__}")
    return result

In [ ]:
# 위 def 셀은 함수를 준비했고, 이 줄에서 실제 /sys와 /proc 파일을 읽습니다.
status = read_system_status()
print(json.dumps(status, ensure_ascii=False, indent=2))

## 2. @tool로 도구의 이름과 설명을 제공합니다

`@tool`은 바로 아래 함수를 LangChain 도구로 바꾸는 데코레이터입니다. 함수 안 첫 문자열인 docstring은 모델이 읽는 도구 설명이 됩니다. `-> str`은 반환값이 문자열임을 나타내는 타입 힌트입니다.

이 도구는 인자가 없으므로 직접 호출할 때 빈 사전 `{}`를 전달합니다. 함수를 정의하는 셀과 실행하는 셀을 구분하세요.


### 사용 전 API 안내 · Tool 명세와 실행

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `tool` 데코레이터 | 함수 위에 `@tool` 작성 | 타입 힌트와 docstring으로 모델에게 보여줄 입력 규격·설명을 만듭니다. `get_jetson_status`는 Tool 객체가 됩니다. |
| Tool 속성 / 호출 | `get_jetson_status.args`; `get_jetson_status.invoke({})` | 입력 규격 사전 / 실제 조회 결과 문자열. 인자가 없어서 빈 사전을 전달합니다. |
| `model.bind_tools` | `bound = model.bind_tools([get_jetson_status])` | 도구 명세를 붙인 호출 객체. 바인딩만으로 Python 함수를 실행하지 않습니다. |
| `AIMessage.tool_calls` | `request.tool_calls` | 도구 요청 목록. 각 항목의 `name`, `args`, `id`를 확인합니다. |
| `ToolMessage` | `ToolMessage(content=결과, tool_call_id=요청id, name=도구명)` | Python 실행 결과를 모델에게 돌려주는 메시지. 반드시 원래 요청 ID와 연결합니다. |

In [ ]:
from langchain_core.tools import tool

@tool
def get_jetson_status() -> str:
    """현재 Jetson의 센서 온도와 메모리를 조회합니다. 측정 시각과 조회 오류도 반환합니다."""
    result = read_system_status()
    # 모델에 돌려줄 수 있도록 Python 사전을 JSON 텍스트로 바꿉니다.
    return json.dumps(result, ensure_ascii=False)

print("도구 이름:", get_jetson_status.name)
print("도구 설명:", get_jetson_status.description)
print("입력 형식:", get_jetson_status.args)


In [ ]:
# 도구 자체의 invoke는 Python 함수를 실행합니다. 여기서는 LLM을 부르지 않습니다.
tool_text = get_jetson_status.invoke({})
print(tool_text)


## 3. bind_tools는 도구 정보를 모델에 전달합니다

모델 응답의 `content`가 비어 있어도 `tool_calls`가 있다면 도구 호출 요청일 수 있습니다. 호출 요청에는 name, args, id가 있습니다. id는 나중에 실행 결과를 원래 요청과 연결합니다.

여기까지 모델은 함수를 실행하지 않았습니다. 실제 조회는 다음 절에서 Python이 수행합니다.

![3. bind_tools는 도구 정보를 모델에 전달합니다](assets/lesson-3-13.svg)


In [ ]:
tool_model = model.bind_tools([get_jetson_status])
messages = [
    SystemMessage("현재 값은 반드시 도구로 조회하세요. 측정하지 않은 항목은 추측하지 마세요."),
    HumanMessage("현재 Jetson의 온도와 메모리를 조회해줘."),
]
request = tool_model.invoke(messages)
print("응답 본문:", request.content)
print("도구 호출 요청:", request.tool_calls)
if not request.tool_calls:
    raise RuntimeError("모델이 도구를 요청하지 않았습니다. 모델 태그·설명·질문을 확인하세요.")


## 4. Python 실행 결과를 ToolMessage로 돌려줍니다

모델의 호출 요청도 대화에 남기고, 각 호출 id에 맞는 실행 결과를 이어 붙입니다. tool_calls는 목록이므로 for로 하나씩 처리합니다.

`try/except`는 도구 실행 오류를 잡아 오류 결과로 돌려주는 문법입니다. 실패를 정상 측정값으로 바꾸지 않습니다. 등록된 도구 이름만 실행합니다.

![4. Python 실행 결과를 ToolMessage로 돌려줍니다](assets/lesson-3-14.svg)


In [ ]:
from langchain_core.messages import ToolMessage

available_tools = {get_jetson_status.name: get_jetson_status}
messages.append(request)  # 모델이 낸 호출 요청을 먼저 기록합니다.
for call in request.tool_calls:
    name = call["name"]
    arguments = call["args"]
    try:
        if name not in available_tools:
            raise ValueError("등록되지 않은 도구입니다.")
        output = available_tools[name].invoke(arguments)
    except Exception as exc:
        output = json.dumps({"error": str(exc)}, ensure_ascii=False)
    # tool_call_id로 어떤 호출에 대한 결과인지 연결합니다.
    messages.append(ToolMessage(content=output, tool_call_id=call["id"], name=name))
    print("실행 도구:", name, "인자:", arguments)
    print("실행 결과:", output)


In [ ]:
# 한 주기 실습이므로 도구를 바인딩하지 않은 model에 결과를 전달해 설명을 받습니다.
final_reply = model.invoke(messages)
print(final_reply.content)
# 확인: 최종 답의 숫자와 센서 이름이 바로 위 ToolMessage에 실제로 있나요?


## 5. TODO · 텍스트 파일을 단어 목록으로 반환하는 Tool

`read_words(filename: str) -> list[str]`을 완성하세요. 먼저 UTF-8 텍스트를 읽고, 공백·줄바꿈 기준으로 `split()`한 목록을 반환합니다. 여기서 단어는 **공백으로 나눈 조각**이며 형태소나 LLM 토큰과 다릅니다. `"안녕, 세계!"`는 `["안녕,", "세계!"]`가 됩니다.

순서: 입력·반환 자료형 정하기 → 파일 읽기 → split → @tool → 모델 없이 invoke 검사입니다. 기본 예제의 기대 결과는 `["Jetson", "상태", "확인", "온도", "메모리", "기록"]`입니다. 경로 검사 코드는 제공하며, 이번에는 지정한 교재 폴더의 작은 파일만 읽습니다.

### 사용 전 API 안내 · 파일 읽기 Tool

| 함수·클래스 | 초기화·사용 | 입력·반환값과 주의점 |
|---|---|---|
| `Path.resolve` / `is_relative_to` | `path = (TEXT_ROOT / filename).resolve()` | 경로를 정규화한 뒤 교재 폴더 내부인지 확인합니다. |
| `read_text` / `str.split` | `path.read_text(encoding="utf-8").split()` | 텍스트 파일 → 문자열 → 공백 기준 단어 목록. |
| `@tool` / `.invoke` | `read_words.invoke({"filename": "sample.txt"})` | 함수 명세를 Tool로 만들고 인자 사전으로 직접 실행합니다. |

In [ ]:
TEXT_ROOT = (ROOT / "04.LLM_v2/data/tool_texts").resolve()

def checked_text_path(filename):
    # 상위 경로(..)와 심볼릭 링크를 해석한 실제 경로를 확인합니다.
    path = (TEXT_ROOT / filename).resolve()
    if not path.is_relative_to(TEXT_ROOT):
        raise ValueError("교재의 tool_texts 폴더 안의 파일만 읽을 수 있습니다.")
    if not path.is_file() or path.suffix != ".txt":
        raise ValueError("존재하는 .txt 파일을 지정하세요.")
    if path.stat().st_size > 100_000:
        raise ValueError("실습에서는 100 KB 이하 파일만 읽습니다.")
    return path

@tool
def read_words(filename: str) -> list[str]:
    """교재 tool_texts 폴더의 UTF-8 텍스트 파일을 공백 기준 단어 목록으로 읽습니다."""
    path = checked_text_path(filename)
    # TODO: 파일 내용을 읽은 뒤 공백·줄바꿈 단위로 나누어 반환하세요.
    raise NotImplementedError("read_text와 split을 사용하세요.")

In [ ]:
# 참고 정답: 다른 Tool 이름으로 제공하므로 학생 Tool을 덮어쓰지 않습니다.
@tool
def read_words_solution(filename: str) -> list[str]:
    """교재 tool_texts 폴더의 UTF-8 텍스트 파일을 공백 기준 단어 목록으로 읽습니다."""
    path = checked_text_path(filename)
    text = path.read_text(encoding="utf-8")  # 파일의 내용을 문자열로 읽습니다.
    words = text.split()  # 인자를 생략하면 연속 공백·탭·줄바꿈을 모두 구분자로 씁니다.
    return words  # 문자열이 아니라 문자열 목록을 반환합니다.

In [ ]:
# 학생 Tool을 먼저 검사합니다. 참고 정답은 위 TODO에 직접 반영하세요.
active_words_tool = read_words
if os.environ.get("DAY2_TODO_VARIANT") == "solution":
    active_words_tool = read_words_solution
words = active_words_tool.invoke({"filename": "sample.txt"})
assert words == ["Jetson", "상태", "확인", "온도", "메모리", "기록"]
print(words)
print("단어 수:", len(words))

## 완료 확인 · 직접 설명해보세요

- [ ] 도구 명세를 읽는 모델과 함수를 실행하는 Python을 구분했다.
- [ ] tool_call_id가 요청과 결과를 연결함을 확인했다.
- [ ] 실제 측정값·조회 실패·모델 추정을 구분했다.

실행 성공과 의미 정답을 구분하세요. 답변이 이상하면 원문을 남기고, 어떤 입력 또는 단계에서 문제가 생겼는지 표시합니다.
